In [1]:
from pathlib import Path
import pandas as pd
import json
import os

In [2]:
top_path = Path(os.path.dirname(os.getcwd()))
data_path = top_path / "data"
insights_path = top_path / "reports" / "insights"

notebooks_path = top_path / "notebooks"
team_data_path = data_path / "processed" / "team_data.parquet"
predictions_path = insights_path / "LightGBM_predictions.parquet"

In [3]:
predictions_df = pd.read_parquet(predictions_path).rename(columns={"result": "prediction"})
team_df = pd.read_parquet(team_data_path)

## Store the accuracy per league in a dictionary

In [4]:
league_data = team_df[["gameid", "side", "league", "result"]]
predictions_df = predictions_df.merge(league_data, on=["gameid", "side"], how="inner", validate="many_to_many")

In [5]:
# Explore accuracy by league
leagues = predictions_df["league"].unique()
accuracies = {}

for league in leagues:
    league_df = predictions_df[predictions_df["league"] == league]
    accuracy = league_df["prediction"] == league_df["result"]
    accuracies[league] = {"count": len(league_df), "accuracy": accuracy.mean()}

accuracies = {k: v for k, v in sorted(accuracies.items(), key=lambda item: item[1]["accuracy"], reverse=True)}

with open(insights_path / "LightGBM_league_accuracies.json", "w") as f:
    json.dump(accuracies, f)

# Specific League Predictions Analysis

In [6]:
# Specific League Analysis
analysis_league = "LEC"
games_data =  team_df[["date", "gameid", "teamname", "opponentteam", "side"]]

lec_df = predictions_df[predictions_df["league"] == analysis_league][["gameid", "side", "league", "prediction", "result"]]
lec_df = games_data.merge(lec_df, on=["gameid", "side"])

lec_df["correct"] = lec_df["prediction"] == lec_df["result"]
lec_df.sort_values(by="date", inplace=True)

lec_df

,date,gameid,teamname,opponentteam,side,league,prediction,result,correct
0,2022-01-15 16:12:38,ESPORTSTMNT04_2090389,Excel Esports,Team BDS,Blue,LEC,0,0,True
1,2022-01-15 16:12:38,ESPORTSTMNT04_2090389,Team BDS,Excel Esports,Red,LEC,0,1,False
2,2022-01-16 17:07:19,ESPORTSTMNT01_2692407,Rogue,Astralis,Blue,LEC,1,1,True
3,2022-01-16 17:07:19,ESPORTSTMNT01_2692407,Astralis,Rogue,Red,LEC,0,0,True
4,2022-01-21 17:09:26,ESPORTSTMNT01_2705072,Astralis,Fnatic,Blue,LEC,0,0,True
...,...,...,...,...,...,...,...,...,...
287,2024-06-10 15:58:00,LOLTMNT04_51028,SK Gaming,GiantX,Red,LEC,1,1,True
288,2024-06-15 16:50:13,LOLTMNT04_52459,MAD Lions KOI,Team Vitality,Blue,LEC,1,0,False
289,2024-06-15 16:50:13,LOLTMNT04_52459,Team Vitality,MAD Lions KOI,Red,LEC,0,1,False
290,2024-06-16 17:34:41,LOLTMNT04_52519,SK Gaming,Fnatic,Blue,LEC,0,0,True
